# Protein Variant-Effect ML Teaching Notebook

This version explains the workflow more explicitly and now includes stacking ensemble learning.


## Why add stacking?

A stacking model is a meta-learner. Instead of trusting one base model, it learns how to combine the strengths of several models.

In this project that means:
- single models still matter because they capture different patterns
- the stack can sometimes turn those different predictions into a stronger final estimate
- if stacking does not win, that is also useful information because it tells us the extra complexity was not worth it


In [ ]:
import json, warnings, joblib, matplotlib.pyplot as plt, pandas as pd, seaborn as sns
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay
from sklearn.model_selection import train_test_split
from protein_variant_ml_project import OUTPUT_DIR, RANDOM_STATE, build_forest_classifier, build_forest_regressor, build_lightgbm_classifier, build_lightgbm_regressor, build_logistic_pipeline, build_modeling_table, build_stacking_classifier, build_stacking_regressor, build_xgboost_classifier, build_xgboost_regressor, classification_metrics, get_feature_lists, get_shap_importance_table, load_dataset, regression_metrics
sns.set_theme(style='whitegrid')


## Step 1: Load and clean the dataset

We keep only single mutants and engineer mutation-based features so the task stays focused on a clean first variant-effect problem.


In [ ]:
raw_df = load_dataset()
modeling_df = build_modeling_table(raw_df)
modeling_df[['mut_type', 'wt_aa', 'position', 'mut_aa', 'substitution_pair', 'label']].head(10)


## Step 2: Split the data once

Using one consistent split keeps model comparison fair.


In [ ]:
train_valid, test_df = train_test_split(modeling_df, test_size=0.2, random_state=RANDOM_STATE, stratify=modeling_df['label'])
train_df, valid_df = train_test_split(train_valid, test_size=0.25, random_state=RANDOM_STATE, stratify=train_valid['label'])
categorical_features, numeric_features = get_feature_lists(modeling_df)
feature_columns = categorical_features + numeric_features
X_train, X_valid, X_test = train_df[feature_columns], valid_df[feature_columns], test_df[feature_columns]
y_train, y_valid, y_test = train_df['label'], valid_df['label'], test_df['label']


## Step 3: The classifier ladder

We compare:
- no-skill baseline
- logistic regression
- random forest
- XGBoost
- LightGBM
- stacking classifier

This lets us see whether the meta-learner beats the best individual model.


In [ ]:
classifiers = {
    'majority_baseline': DummyClassifier(strategy='most_frequent'),
    'logistic_regression': build_logistic_pipeline(categorical_features, numeric_features),
    'random_forest': build_forest_classifier(categorical_features, numeric_features),
    'xgboost': build_xgboost_classifier(categorical_features, numeric_features),
    'lightgbm': build_lightgbm_classifier(categorical_features, numeric_features),
    'stacking_classifier': build_stacking_classifier(categorical_features, numeric_features),
}
classification_results, fitted_classifiers = {}, {}
for name, model in classifiers.items():
    model.fit(X_train, y_train)
    fitted_classifiers[name] = model
    classification_results[f'{name}_valid'] = classification_metrics(model, X_valid, y_valid)
pd.DataFrame(classification_results).T.round(4)


## Step 4: Pick the best classifier

We still pick by validation performance. That keeps the test set honest.


In [ ]:
best_model_name = max(['logistic_regression', 'random_forest', 'xgboost', 'lightgbm', 'stacking_classifier'], key=lambda n: (classification_results[f'{n}_valid']['f1'], classification_results[f'{n}_valid']['roc_auc']))
best_model = fitted_classifiers[best_model_name]
classification_results[f'{best_model_name}_test'] = classification_metrics(best_model, X_test, y_test)
classification_results['majority_baseline_test'] = classification_metrics(fitted_classifiers['majority_baseline'], X_test, y_test)
pd.DataFrame(classification_results).T.round(4)


## Step 5: Visualize classification behavior


In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    probs = best_model.predict_proba(X_test)[:, 1]
    preds = best_model.predict(X_test)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=axes[0], colorbar=False)
RocCurveDisplay.from_predictions(y_test, probs, ax=axes[1])
PrecisionRecallDisplay.from_predictions(y_test, probs, ax=axes[2])
plt.tight_layout(); plt.show()
best_model_name


## Step 6: How to interpret a stack

A stack is harder to explain directly because the final prediction is built from several models.

A practical strategy is:
- keep a simple interpretable baseline such as logistic regression
- inspect permutation importance on the winning model
- use SHAP on a strong tree-based component if the stack wins

That is what we do below.


In [ ]:
logistic_model = fitted_classifiers['logistic_regression']
feature_names = logistic_model.named_steps['preprocessor'].get_feature_names_out()
coeffs = logistic_model.named_steps['model'].coef_[0]
pd.DataFrame({'feature': feature_names, 'coefficient': coeffs, 'abs_coefficient': abs(coeffs)}).sort_values('abs_coefficient', ascending=False).head(15)


In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    perm = permutation_importance(best_model, X_test, y_test, n_repeats=20, random_state=RANDOM_STATE, scoring='f1', n_jobs=1)
pd.DataFrame({'feature': X_test.columns, 'importance_mean': perm.importances_mean}).sort_values('importance_mean', ascending=False).head(15)


In [ ]:
shap_target_model = fitted_classifiers['lightgbm'] if best_model_name == 'stacking_classifier' else best_model
shap_df = get_shap_importance_table(shap_target_model, X_test) if hasattr(shap_target_model, 'named_steps') else None
shap_df.head(15) if shap_df is not None else 'SHAP skipped'


## Step 7: The regression ladder

We do the same thing for continuous prediction. This is where stacking can be especially useful because different regressors often make different numeric mistakes.


In [ ]:
y_train_reg, y_valid_reg, y_test_reg = train_df['ddG_ML_float'], valid_df['ddG_ML_float'], test_df['ddG_ML_float']
regressors = {
    'dummy_regressor': DummyRegressor(strategy='mean'),
    'random_forest_regressor': build_forest_regressor(categorical_features, numeric_features),
    'xgboost_regressor': build_xgboost_regressor(categorical_features, numeric_features),
    'lightgbm_regressor': build_lightgbm_regressor(categorical_features, numeric_features),
    'stacking_regressor': build_stacking_regressor(categorical_features, numeric_features),
}
regression_results, fitted_regressors = {}, {}
for name, model in regressors.items():
    model.fit(X_train, y_train_reg)
    fitted_regressors[name] = model
    regression_results[f'{name}_valid'] = regression_metrics(model, X_valid, y_valid_reg)
best_regressor_name = min(['random_forest_regressor', 'xgboost_regressor', 'lightgbm_regressor', 'stacking_regressor'], key=lambda n: (regression_results[f'{n}_valid']['mae'], regression_results[f'{n}_valid']['rmse']))
regression_results[f'{best_regressor_name}_test'] = regression_metrics(fitted_regressors[best_regressor_name], X_test, y_test_reg)
regression_results['dummy_regressor_test'] = regression_metrics(fitted_regressors['dummy_regressor'], X_test, y_test_reg)
pd.DataFrame(regression_results).T.round(4)


## Step 8: What stacking tells us

If the stacking regressor wins, that suggests the base regressors learned complementary information.

If it loses, that tells us the simpler models were already enough. Either way, this is a useful lesson in ensemble learning.


In [ ]:
joblib.dump(best_model, OUTPUT_DIR / 'best_classifier_from_teaching_notebook.joblib')
joblib.dump(fitted_regressors[best_regressor_name], OUTPUT_DIR / 'regressor_from_teaching_notebook.joblib')
summary = {'classification': classification_results, 'regression': regression_results, 'best_model_name': best_model_name, 'best_regressor_name': best_regressor_name}
(OUTPUT_DIR / 'teaching_notebook_metrics_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
summary
